In [1]:
import pandas as pd
sales = pd.read_csv('../data/sales.csv')

In [2]:
sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"])


In [3]:
customer_sales = sales.dropna(
    subset=["Customer ID"]
).copy()

In [4]:
customer_summary = (
    customer_sales
    .groupby("Customer ID")
    .agg(
        Revenue=("Revenue", "sum"),
        Orders=("Invoice", "nunique"),
        Units=("Quantity", "sum"),
        First_Purchase=("InvoiceDate", "min"),
        Last_Purchase=("InvoiceDate", "max")
    )
    .reset_index()
)


In [5]:
# Average Order Value
customer_summary["AOV"] = (
    customer_summary["Revenue"] /
    customer_summary["Orders"]
)


print("Customer Summary:")
print(customer_summary.head(10))

print("Shape:")
print(customer_summary.shape)

Customer Summary:
   Customer ID   Revenue  Orders  Units      First_Purchase  \
0      12346.0    372.86      11     70 2009-12-14 08:34:00   
1      12347.0   1323.32       2    828 2010-10-31 14:20:00   
2      12348.0    222.16       1    373 2010-09-27 14:59:00   
3      12349.0   2671.14       3    993 2010-04-29 13:20:00   
4      12351.0    300.93       1    261 2010-11-29 15:23:00   
5      12352.0    343.80       2    188 2010-11-12 10:20:00   
6      12353.0    317.76       1    192 2010-10-27 12:44:00   
7      12355.0    488.21       1    303 2010-05-21 11:59:00   
8      12356.0   3560.30       3   1825 2010-10-11 09:42:00   
9      12357.0  12079.99       2   3879 2010-11-16 10:05:00   

        Last_Purchase          AOV  
0 2010-06-28 13:53:00    33.896364  
1 2010-12-07 14:57:00   661.660000  
2 2010-09-27 14:59:00   222.160000  
3 2010-10-28 08:23:00   890.380000  
4 2010-11-29 15:23:00   300.930000  
5 2010-11-29 10:07:00   171.900000  
6 2010-10-27 12:44:00   317.7

In [6]:
top_customers = (
    customer_summary
    .sort_values("Revenue", ascending=False)
    .head(10)
)

print("Top 10 Customers by Revenue:")
print(top_customers)

Top 10 Customers by Revenue:
      Customer ID    Revenue  Orders   Units      First_Purchase  \
4185      18102.0  349164.35      89  124216 2009-12-01 09:24:00   
1638      14646.0  248396.50      78  170342 2009-12-02 16:52:00   
1270      14156.0  196549.74     102  108105 2009-12-01 12:30:00   
1842      14911.0  152121.22     205   69709 2009-12-01 11:41:00   
939       13694.0  131443.19      94  125893 2009-12-04 15:26:00   
3746      17511.0   84541.17      31   55107 2009-12-02 10:52:00   
1953      15061.0   83284.38      86   51791 2009-12-01 12:18:00   
3130      16684.0   80489.21      27   54555 2009-12-07 12:56:00   
3179      16754.0   65500.07      29   63551 2010-03-08 11:32:00   
4067      17949.0   60117.60      74   30112 2009-12-02 11:07:00   

           Last_Purchase          AOV  
4185 2010-12-09 13:44:00  3923.194944  
1638 2010-11-30 16:28:00  3184.570513  
1270 2010-12-03 11:48:00  1926.958235  
1842 2010-12-09 12:17:00   742.054732  
939  2010-12-01 12:12:

In [7]:
top_orders = (
    customer_summary
    .sort_values("Orders", ascending=False)
    .head(10)
)

print("Top 10 Customers by Number of Orders:")
print(top_orders)

Top 10 Customers by Number of Orders:
      Customer ID    Revenue  Orders   Units      First_Purchase  \
1842      14911.0  152121.22     205   69709 2009-12-01 11:41:00   
3998      17850.0   51208.87     155   21052 2009-12-05 12:28:00   
251       12748.0   22457.90     144   13110 2009-12-04 17:31:00   
2135      15311.0   55942.74     121   32784 2009-12-01 11:21:00   
506       13089.0   57885.45     109   29130 2009-12-02 15:44:00   
1607      14606.0   18482.10     102    9287 2009-12-03 12:40:00   
1270      14156.0  196549.74     102  108105 2009-12-01 12:30:00   
939       13694.0  131443.19      94  125893 2009-12-04 15:26:00   
3991      17841.0   29562.02      91   14332 2009-12-02 15:41:00   
4185      18102.0  349164.35      89  124216 2009-12-01 09:24:00   

           Last_Purchase          AOV  
1842 2010-12-09 12:17:00   742.054732  
3998 2010-12-02 15:27:00   330.379806  
251  2010-12-09 13:22:00   155.957639  
2135 2010-12-09 14:15:00   462.336694  
506  2010-12-

In [8]:
## one-time vs repeat customers


customer_type = (
    customer_summary["Orders"]
    .apply(
        lambda x: "One-time"
        if x == 1
        else "Repeat"
    )
    .value_counts()
)

print("Customer Type:")
print(customer_type)

Customer Type:
Orders
Repeat      2893
One-time    1421
Name: count, dtype: int64


In [9]:
customer_type_percentage = (
    customer_type /
    customer_type.sum() * 100
)

print("Customer Type Percentage:")
print(customer_type_percentage)

Customer Type Percentage:
Orders
Repeat      67.060732
One-time    32.939268
Name: count, dtype: float64


In [11]:
customer_summary["Customer_Type"] = (
    customer_summary["Orders"]
    .apply(
        lambda x: "One-time"
        if x == 1
        else "Repeat"
    )
)

In [12]:
customer_type_revenue = (
    customer_summary
    .groupby("Customer_Type")
    .agg(
        Customers=("Customer ID", "count"),
        Revenue=("Revenue", "sum"),
        Orders=("Orders", "sum")
    )
    .reset_index()
)

print("Revenue by Customer Type:")
print(customer_type_revenue)

Revenue by Customer Type:
  Customer_Type  Customers      Revenue  Orders
0      One-time       1421   496304.474    1421
1        Repeat       2893  8301929.270   17794


In [13]:
print("Customer Revenue Statistics:")

print(
    customer_summary["Revenue"].describe()
)

Customer Revenue Statistics:
count      4314.000000
mean       2039.460766
std        8909.797773
min           0.000000
25%         307.105000
50%         700.405000
75%        1713.297500
max      349164.350000
Name: Revenue, dtype: float64


In [15]:
zero_revenue_customers = customer_summary[
    customer_summary["Revenue"] == 0
]

print(
    "Customers with zero revenue:",
    len(zero_revenue_customers)
)

Customers with zero revenue: 2


In [16]:
print(
    zero_revenue_customers.head()
)

      Customer ID  Revenue  Orders  Units      First_Purchase  \
1233      14103.0      0.0       1      5 2010-02-12 14:58:00   
1775      14827.0      0.0       1      5 2010-02-12 15:47:00   

           Last_Purchase  AOV Customer_Type  
1233 2010-02-12 14:58:00  0.0      One-time  
1775 2010-02-12 15:47:00  0.0      One-time  
